# 03 · Train five baseline models
Fit the baseline pipelines with five-fold stratified group cross-validation on the training set. Class weighting gives more influence to classes with fewer training examples. XGBoost gets equivalent per-row weights recalculated on each fit.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display
from sklearn.model_selection import cross_validate
from src.train import model_candidates
from src.preprocessing import METRICS, five_folds, make_pipeline, make_split

X_train, _, y_train, _ = make_split()
scoring = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
folds = five_folds(X_train, y_train)
rows = []
for name, model in model_candidates().items():
    scores = cross_validate(make_pipeline(model), X_train, y_train,
                            scoring=scoring, cv=folds, error_score="raise")
    row = {"model": name}
    row.update({metric: scores[f"test_{metric}"].mean() for metric in scoring})
    row["f1_macro_std"] = scores["test_f1_macro"].std()
    rows.append(row)

baseline = pd.DataFrame(rows).sort_values("f1_macro", ascending=False)
baseline.to_csv(METRICS / "baseline_cv.csv", index=False)
display(baseline.round(4))

,model,accuracy,precision_macro,recall_macro,f1_macro,f1_macro_std
2,SVM,0.5031,0.4788,0.4805,0.4741,0.0283
0,Logistic Regression,0.4828,0.4635,0.4617,0.4596,0.0256
1,Random Forest,0.5062,0.4688,0.4605,0.4530,0.0402
4,LightGBM,0.4656,0.4365,0.4340,0.4329,0.0442
3,XGBoost,0.4500,0.4167,0.4189,0.4161,0.0518


Macro averages give Low, Medium and High equal weight. These baseline scores summarize training-fold validation. Notebook 05 reports held-out metrics.